# J15b — Libellés sans préfixes techniques et âges explicites

Ce correctif complémentaire s'exécute **après la génération J15b et le précédent correctif des libellés et unités**.
Il modifie uniquement `LIBELLE` dans le catalogue existant, puis met à jour son empreinte dans le manifeste.
Les unités, définitions, codes de variables, identifiants, sources, valeurs et statuts de validation sont conservés.

Exemples : `J13e part 1529 pct` → `Part 15-29 ans pct` ; `Population 30 à 44 ans ans 2022` → `Population 30-44 ans 2022`.
Le correctif garde les mots du libellé déjà préparé : il complète le précédent travail de traduction.
Les bornes ouvertes deviennent `75 ans et plus` et `moins de 30 ans`.

Extraire le ZIP à la racine du projet pour disposer de `app/libelles_affichage.py`.
Une sauvegarde du catalogue et du manifeste sera créée avant modification.

## 1 — Imports et racine du projet

Si le projet n'est pas détecté, renseigner `RACINE_MANUELLE` avec son chemin.

In [2]:
from pathlib import Path
import os
import sys
import json
import csv
import importlib.util

RACINE_MANUELLE = ""  # Exemple : r"C:/Users/almou/OneDrive/GeoMarketing_IDF"
candidats = ([Path(RACINE_MANUELLE)] if RACINE_MANUELLE else
    ([Path(os.environ["GEOMARKETING_RACINE"])] if os.environ.get("GEOMARKETING_RACINE") else
     [Path.cwd(), *Path.cwd().parents, Path(r"C:/Users/almou/OneDrive/GeoMarketing_IDF")]))
RACINE = next((p.resolve() for p in candidats if (p / "app/libelles_affichage.py").is_file()), None)
if RACINE is None:
    raise FileNotFoundError("Extraire le ZIP à la racine du projet puis renseigner RACINE_MANUELLE si nécessaire.")
module_path = RACINE / "app/libelles_affichage.py"
spec = importlib.util.spec_from_file_location("correctif_j15b_libelles_affichage", module_path)
correctif = importlib.util.module_from_spec(spec)
spec.loader.exec_module(correctif)
print("Racine :", RACINE)
print("Correctif :", correctif.VERSION)

Racine : C:\Users\almou\OneDrive\GeoMarketing_IDF
Correctif : 2026-10-22-prefixes-ages-v1


## 2 — Choisir les exports J15b à corriger

Le dossier local habituel est `data/app/j15b`. Le dépôt déployé peut utiliser `app/data/app/j15b`.
Le notebook prend le premier dossier existant de la liste ci-dessous, puis affiche son chemin exact.
Si tu prépares les exports locaux, garde `data/app/j15b`, puis publie les deux fichiers modifiés dans ton dossier GitHub.
Pour cibler un autre dossier, renseigner `DOSSIER_DETAIL_MANUEL` et, si nécessaire, `DOSSIER_BASE_MANUEL`.

In [3]:
DOSSIER_DETAIL_MANUEL = ""  # Exemple : "data/app/j15b" ou "app/data/app/j15b"
DOSSIER_BASE_MANUEL = ""    # Vide : prend le dossier j15 voisin

def chemin_projet(valeur):
    p = Path(valeur)
    return p if p.is_absolute() else RACINE / p

detail_config = DOSSIER_DETAIL_MANUEL or os.environ.get("GEOMARKETING_APP_DETAIL", "")
possibles = ([chemin_projet(detail_config)] if detail_config else
             [RACINE / "data/app/j15b", RACINE / "app/data/app/j15b"])
DOSSIER_DETAIL = next((p.resolve() for p in possibles if (p / "manifest.json").is_file()), None)
if DOSSIER_DETAIL is None:
    raise FileNotFoundError("Exports J15b introuvables : renseigner DOSSIER_DETAIL_MANUEL.")
DOSSIER_BASE = (chemin_projet(DOSSIER_BASE_MANUEL).resolve() if DOSSIER_BASE_MANUEL else
                DOSSIER_DETAIL.parent / "j15")
DOSSIER_AUDIT = RACINE / "data/interim/j15b/libelles_prefixes_ages"
print("J15b corrigé :", DOSSIER_DETAIL)
print("J15a associé :", DOSSIER_BASE)
if len([p for p in possibles if (p / "manifest.json").is_file()]) > 1:
    print("Plusieurs copies existent. Le correctif utilisera le dossier affiché ci-dessus.")
manifest_avant = correctif.verifier_exports(DOSSIER_DETAIL, DOSSIER_BASE)
print("Exports J15a et J15b synchronisés ✅")

J15b corrigé : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\app\j15b
J15a associé : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\app\j15
Exports J15a et J15b synchronisés ✅


## 3 — Prévisualiser les changements

Les références J12, J13a, J13b, J13c, J13d, J13e, J14, J15b, etc. disparaissent des libellés.
Les années comme 2022 et les quantités comme « pour 10 000 habitants » restent présentes.
Les pondérations de scénarios telles que 70/30 ne sont pas interprétées comme des âges.

In [4]:
champs_avant, lignes_avant, separateur = correctif._lire_catalogue(DOSSIER_DETAIL / "catalogue.csv")
lignes_apres, changements = correctif.nettoyer_catalogue(lignes_avant)
print("Variables :", len(lignes_avant))
print("Libellés à corriger :", len(changements))
for ligne in changements[:30]:
    print(ligne["COLONNE"])
    print("  Avant :", ligne["LIBELLE_AVANT"])
    print("  Après :", ligne["LIBELLE_APRES"])

Variables : 1348
Libellés à corriger : 505
ORIGINE_SELECTION_J12
  Avant : Origine selection j12
  Après : Origine selection
J13A_COMMUNE_PRESELECTIONNEE_J12
  Avant : J13a commune preselectionnee j12
  Après : Commune preselectionnee
J13A_DANS_FOCUS
  Avant : J13a dans focus
  Après : Dans focus
J13A_SURFACE_KM2
  Avant : J13a surface km2
  Après : Surface km2
P22_POP0014
  Avant : P22 pop0014
  Après : P22 pop0-14 ans
P22_POP1529
  Avant : P22 pop1529
  Après : P22 pop15-29 ans
P22_POP3044
  Avant : P22 pop3044
  Après : P22 pop30-44 ans
P22_POP4559
  Avant : P22 pop4559
  Après : P22 pop45-59 ans
P22_POP6074
  Avant : P22 pop6074
  Après : P22 pop60-74 ans
P22_POP75P
  Avant : P22 pop75p
  Après : P22 pop75 ans et plus
J13A_POPULATION
  Avant : J13a population
  Après : Population
J13A_POPULATION_HOMMES
  Avant : J13a population hommes
  Après : Population hommes
J13A_POPULATION_FEMMES
  Avant : J13a population femmes
  Après : Population femmes
J13A_POP_0014
  Avant : J13a pop 0014

## 4 — Appliquer le correctif et synchroniser le manifeste

Le journal et les sauvegardes vont dans `data/interim/j15b/libelles_prefixes_ages/`.
Tous les autres exports restent identiques. Les libellés validés ou saisis manuellement reçoivent
les mêmes corrections de présentation ; leur définition, leur unité et leur statut sont conservés.

In [5]:
bilan = correctif.appliquer_correctif(DOSSIER_DETAIL, DOSSIER_BASE, DOSSIER_AUDIT)
print(json.dumps(bilan, ensure_ascii=False, indent=2))

{
  "nb_variables": 1348,
  "nb_modifications": 505,
  "journal": "C:\\Users\\almou\\OneDrive\\GeoMarketing_IDF\\data\\interim\\j15b\\libelles_prefixes_ages\\corrections_prefixes_ages_j15b.csv",
  "sauvegarde": "C:\\Users\\almou\\OneDrive\\GeoMarketing_IDF\\data\\interim\\j15b\\libelles_prefixes_ages\\sauvegardes\\20261022T182850_740285Z"
}


## 5 — Vérifier le résultat

In [6]:
manifest_apres = correctif.verifier_exports(DOSSIER_DETAIL, DOSSIER_BASE)
champs_apres, resultat, _ = correctif._lire_catalogue(DOSSIER_DETAIL / "catalogue.csv")
assert champs_avant == champs_apres
assert len(resultat) == len(lignes_avant)
for avant, apres in zip(lignes_avant, resultat):
    assert {k:v for k,v in avant.items() if k != "LIBELLE"} == {
        k:v for k,v in apres.items() if k != "LIBELLE"}
assert not correctif.nettoyer_catalogue(resultat)[1], "Le nettoyage doit être stable à la seconde exécution."
for entree in manifest_avant["fichiers"]:
    if entree["nom"] != "catalogue.csv":
        nouvelle = next(e for e in manifest_apres["fichiers"] if e["nom"] == entree["nom"])
        assert nouvelle == entree
print("Libellés corrigés ; autres métadonnées et données conservées ✅")
print("Exports J15a et J15b synchronisés ✅")
print("Catalogue à publier :", DOSSIER_DETAIL / "catalogue.csv")
print("Manifeste à publier :", DOSSIER_DETAIL / "manifest.json")

Libellés corrigés ; autres métadonnées et données conservées ✅
Exports J15a et J15b synchronisés ✅
Catalogue à publier : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\app\j15b\catalogue.csv
Manifeste à publier : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\app\j15b\manifest.json


## 6 — Mettre à jour GitHub et Streamlit

1. Dans ton organisation actuelle, publier **ensemble** le `catalogue.csv` et le `manifest.json` corrigés
   dans `app/data/app/j15b/`. Prendre les fichiers du dossier affiché à l'étape 2.
2. Ajouter `app/libelles_affichage.py` et ce notebook dans le dépôt.
3. Vérifier l'affichage après le redéploiement Streamlit. Si nécessaire, redémarrer l'application pour vider ses caches.
4. Après toute nouvelle génération complète de J15b, réexécuter ce notebook avant de publier les exports.

Commit proposé : `fix: supprimer les préfixes techniques et clarifier les tranches d'âge`

Les colonnes `ID`, `COLONNE`, `SOURCE`, `ANNEE`, `UNITE` et les fichiers de données ne sont pas renommés.
La version de schéma du manifeste reste 1 pour J15b.